#CS-245 Machine Learning Project
## Phase 2: Supervised Model Development & Evaluation

---

**Group Members:** 
Zayna Qasim, 
Sukaina Nasir, 
Hamna Shah 


**Course:** CS-245 — Machine Learning

---

Okay so in Phase 1 we cleaned the dataset and did the EDA. Now in this notebook we're actually building the machine learning models.

We're doing a **regression task** — predicting property prices in Pakistan (in Lakhs PKR) based on features like area, bedrooms, bathrooms, city, and property type.

### Models we're implementing:
1. **Linear Regression** — simple baseline, good starting point
2. **Random Forest Regressor** — our main model, handles non-linearity well
3. **Gradient Boosting Regressor** — bonus third model for comparison

We'll evaluate all three using RMSE, MAE, and R² and then compare them at the end.

>**Note:** Make sure you've run Phase 1 first and have `processed_data.csv` in your Colab files. Upload it before running this notebook.


---
## Step 0 — Imports

Same libraries as before plus sklearn model stuff.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# Models
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Preprocessing & Splitting
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Plot styling
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#f8f9fa",
    "axes.grid":        True,
    "grid.color":       "#e0e0e0",
    "grid.linestyle":   "--",
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
})
PALETTE = ["#2E86AB", "#F18F01", "#A23B72", "#44BBA4"]

print("All good, libraries loaded.")


---
## Step 1 — Load the Processed Dataset

We're loading the cleaned CSV from Phase 1. No need to redo all that messy parsing again.


In [ ]:
df = pd.read_csv("processed_data.csv")   # make sure this file is uploaded

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
df.head()


In [ ]:
# Quick check — make sure there are no NaNs left
print("Missing values:")
print(df.isnull().sum())
print(f"\nTarget (price_lakh) stats:")
print(df["price_lakh"].describe().round(2))


---
## Step 2 — Define Features and Target

We're using 6 features to predict the price:

| Feature | What it represents |
|---------|-------------------|
| `area_marla` | Property size (converted to Marla in Phase 1) |
| `bedroom` | Number of bedrooms |
| `bath` | Number of bathrooms |
| `type_enc` | Property type encoded (House=3, Flat=1, etc.) |
| `purpose_enc` | For Sale=1, For Rent=0 |
| `city_enc` | City encoded as integer |

**Target:** `price_lakh` — the property price in Lakhs PKR


In [ ]:
FEATURES = ["area_marla", "bedroom", "bath", "type_enc", "purpose_enc", "city_enc"]
TARGET   = "price_lakh"

X = df[FEATURES]
y = df[TARGET]

print(f"Feature matrix shape : {X.shape}")
print(f"Target vector shape  : {y.shape}")
print(f"\nPrice range: {y.min():.1f} – {y.max():.1f} Lakhs")


---
## Step 3 — Train/Test Split

We split the data into **80% training** and **20% testing**. The model learns from training data and we evaluate it on the test data it has never seen before.

We use `random_state=42` so results are reproducible every time we run the notebook.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"\nThat's an 80/20 split — {len(X_train)/len(X)*100:.1f}% / {len(X_test)/len(X)*100:.1f}%")


---
## Step 4 — Feature Scaling (for Linear Regression)

Linear Regression is sensitive to the scale of features. For example, `area_marla` can be up to 400+ while `purpose_enc` is just 0 or 1. Without scaling, the model gives too much weight to larger numbers.

We use **StandardScaler** which transforms each feature to have mean=0 and std=1.

> Important: We fit the scaler **only on training data** and then apply it to test data. If we fit on the full dataset it would be *data leakage* — the model would indirectly know information about the test set.

Random Forest and Gradient Boosting don't need scaling (they're tree-based), so we keep both versions.


In [ ]:
scaler = StandardScaler()

# Fit on train only — NEVER fit on test data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)   # just transform, don't re-fit

print("Scaling done.")
print(f"X_train mean (after scaling) ≈ {X_train_scaled.mean():.4f}  (should be ~0)")
print(f"X_train std  (after scaling) ≈ {X_train_scaled.std():.4f}   (should be ~1)")


---
## Step 5 — Evaluation Metrics

Before training, we set up a helper function to calculate all our metrics in one go.

For regression problems, we use:

| Metric | Formula | What it tells us |
|--------|---------|-----------------|
| **RMSE** | √(mean of squared errors) | Average error in same units as price (Lakhs). Penalises big errors more. |
| **MAE** | mean of absolute errors | Average absolute error — easier to interpret |
| **R²** | 1 - SS_res/SS_tot | How much variance the model explains. 1.0 = perfect, 0 = as good as just predicting the mean |



In [ ]:
def evaluate_model(name, y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    
    print(f"\n{'='*40}")
    print(f"  Model   : {name}")
    print(f"{'='*40}")
    print(f"  RMSE    : {rmse:.2f} Lakhs")
    print(f"  MAE     : {mae:.2f} Lakhs")
    print(f"  R²      : {r2:.4f}")
    print(f"{'='*40}")
    
    return {"Model": name, "RMSE": round(rmse, 2), "MAE": round(mae, 2), "R²": round(r2, 4)}

results = []   # we'll store all model results here
predictions = {}  # store predictions for plotting later


---
## Model 1 — Linear Regression

Linear Regression is the simplest model we can use. It assumes a straight-line relationship between features and the target:

$$\hat{price} = w_1 \times area + w_2 \times bedroom + w_3 \times bath + ... + b$$

We're using it as a **baseline** — if our other models can't beat this, something is wrong.

We expect it to underperform on this dataset because property prices in Pakistan have a lot of non-linear relationships (e.g., a house in DHA Lahore is disproportionately more expensive than its area alone would suggest).


In [ ]:
# Train Linear Regression
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)

# Predict on test set
y_pred_lr = lr_model.predict(X_test_scaled)

# Evaluate
lr_results = evaluate_model("Linear Regression", y_test, y_pred_lr)
results.append(lr_results)
predictions["Linear Regression"] = y_pred_lr


In [ ]:
# Look at the coefficients — which features does LR think matter most?
coeff_df = pd.DataFrame({
    "Feature":     FEATURES,
    "Coefficient": lr_model.coef_
}).sort_values("Coefficient", ascending=False)

print("Feature Coefficients (how much each feature affects predicted price):")
print(coeff_df.to_string(index=False))
print(f"\nIntercept: {lr_model.intercept_:.2f}")


---
## Model 2 — Random Forest Regressor

Random Forest builds many decision trees (we used 100) and averages their predictions. It's great at capturing non-linear relationships which is exactly what we need here.

**Why Random Forest for property prices?**
- Price doesn't scale linearly with area — a 20 Marla house isn't always twice the price of a 10 Marla one
- Location (city_enc) has a non-linear effect — some cities are just way more expensive
- Random Forest naturally handles these complex patterns

We don't need to scale features for this model since trees split based on thresholds, not distances.


In [ ]:
# Train Random Forest — this might take 30-60 seconds
print("Training Random Forest... (100 trees, please wait)")

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1          # use all CPU cores to speed it up
)
rf_model.fit(X_train, y_train)   # no scaling needed for RF

# Predict
y_pred_rf = rf_model.predict(X_test)

# Evaluate
rf_results = evaluate_model("Random Forest", y_test, y_pred_rf)
results.append(rf_results)
predictions["Random Forest"] = y_pred_rf

print("\nDone!")


In [ ]:
# Feature Importance — what does RF think matters most?
feat_imp = pd.DataFrame({
    "Feature":    FEATURES,
    "Importance": rf_model.feature_importances_
}).sort_values("Importance", ascending=False)

print("Feature Importances (Random Forest):")
print(feat_imp.to_string(index=False))


---
## Model 3 — Gradient Boosting Regressor (Bonus)

Gradient Boosting also builds trees but in a different way — instead of building them independently (like Random Forest), it builds them **sequentially**. Each new tree tries to fix the errors the previous trees made.

It usually gives better accuracy but takes longer to train and is harder to tune.

We're including it as our third model to strengthen the comparative analysis section of the report.


In [ ]:
# Train Gradient Boosting
print("Training Gradient Boosting... (this one's slower, bear with us)")

gb_model = GradientBoostingRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42
)
gb_model.fit(X_train, y_train)

# Predict
y_pred_gb = gb_model.predict(X_test)

# Evaluate
gb_results = evaluate_model("Gradient Boosting", y_test, y_pred_gb)
results.append(gb_results)
predictions["Gradient Boosting"] = y_pred_gb

print("\nDone!")


---
##  Step 6 — Cross-Validation

A single train/test split might get lucky or unlucky depending on which rows ended up in the test set. **5-Fold Cross-Validation** splits the data into 5 parts, trains on 4, tests on 1, and repeats 5 times. We average the scores.

This gives a more reliable picture of how well each model actually generalises.


In [ ]:
print("Running 5-Fold Cross-Validation on all models...")
print("(This will take a minute or two)\n")

cv_results = {}

# Linear Regression CV
lr_cv = cross_val_score(LinearRegression(), X_train_scaled, y_train,
                         cv=5, scoring="r2")
cv_results["Linear Regression"] = lr_cv
print(f"Linear Regression  — CV R² scores: {lr_cv.round(3)}")
print(f"                     Mean: {lr_cv.mean():.4f}  |  Std: {lr_cv.std():.4f}\n")

# Random Forest CV
rf_cv = cross_val_score(RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1),
                         X_train, y_train, cv=5, scoring="r2")
cv_results["Random Forest"] = rf_cv
print(f"Random Forest      — CV R² scores: {rf_cv.round(3)}")
print(f"                     Mean: {rf_cv.mean():.4f}  |  Std: {rf_cv.std():.4f}\n")

# Gradient Boosting CV
gb_cv = cross_val_score(GradientBoostingRegressor(n_estimators=50, random_state=42),
                         X_train, y_train, cv=5, scoring="r2")
cv_results["Gradient Boosting"] = gb_cv
print(f"Gradient Boosting  — CV R² scores: {gb_cv.round(3)}")
print(f"                     Mean: {gb_cv.mean():.4f}  |  Std: {gb_cv.std():.4f}")


---
##  Step 7 — Comparative Analysis

Now let's compare all three models side by side. This is one of the most important parts for the report — we need to clearly explain which model performed best and why.


In [ ]:
results_df = pd.DataFrame(results)
results_df["CV R² Mean"] = [
    cv_results["Linear Regression"].mean().round(4),
    cv_results["Random Forest"].mean().round(4),
    cv_results["Gradient Boosting"].mean().round(4)
]
results_df["CV R² Std"] = [
    cv_results["Linear Regression"].std().round(4),
    cv_results["Random Forest"].std().round(4),
    cv_results["Gradient Boosting"].std().round(4)
]

print("=" * 70)
print("MODEL COMPARISON TABLE")
print("=" * 70)
print(results_df.to_string(index=False))
print("=" * 70)
print("\nLower RMSE/MAE = better  |  Higher R² = better")


---
## Step 8 — Visualisations

### Plot 1: Model Comparison Bar Chart
Let's visualise the comparison — easier to see than a table.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Model Performance Comparison", fontsize=14, fontweight="bold")

models  = results_df["Model"]
colors  = [PALETTE[0], PALETTE[1], PALETTE[2]]

# RMSE
axes[0].bar(models, results_df["RMSE"], color=colors, edgecolor="white", width=0.5)
axes[0].set_title("RMSE (lower is better)")
axes[0].set_ylabel("RMSE (Lakhs)")
axes[0].tick_params(axis="x", rotation=15)
for i, v in enumerate(results_df["RMSE"]):
    axes[0].text(i, v + 1, str(v), ha="center", fontsize=10, fontweight="bold")

# MAE
axes[1].bar(models, results_df["MAE"], color=colors, edgecolor="white", width=0.5)
axes[1].set_title("MAE (lower is better)")
axes[1].set_ylabel("MAE (Lakhs)")
axes[1].tick_params(axis="x", rotation=15)
for i, v in enumerate(results_df["MAE"]):
    axes[1].text(i, v + 1, str(v), ha="center", fontsize=10, fontweight="bold")

# R²
axes[2].bar(models, results_df["R²"], color=colors, edgecolor="white", width=0.5)
axes[2].set_title("R² Score (higher is better)")
axes[2].set_ylabel("R² Score")
axes[2].set_ylim(0, 1.05)
axes[2].tick_params(axis="x", rotation=15)
for i, v in enumerate(results_df["R²"]):
    axes[2].text(i, v + 0.01, str(v), ha="center", fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("plot8_model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


### Plot 2: Actual vs Predicted — All 3 Models

This scatter plot shows how close predictions are to actual prices. A perfect model would have all points on the red diagonal line.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Actual vs Predicted Prices", fontsize=14, fontweight="bold")

model_preds = [
    ("Linear Regression",  predictions["Linear Regression"],  PALETTE[0]),
    ("Random Forest",      predictions["Random Forest"],       PALETTE[1]),
    ("Gradient Boosting",  predictions["Gradient Boosting"],   PALETTE[2]),
]

for ax, (name, preds, color) in zip(axes, model_preds):
    ax.scatter(y_test, preds, alpha=0.3, s=10, color=color, edgecolors="none")
    
    # Perfect prediction line
    min_val = min(y_test.min(), preds.min())
    max_val = max(y_test.max(), preds.max())
    ax.plot([min_val, max_val], [min_val, max_val], "r--", linewidth=1.5, label="Perfect Prediction")
    
    ax.set_xlabel("Actual Price (Lakhs)")
    ax.set_ylabel("Predicted Price (Lakhs)")
    ax.set_title(name)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("plot9_actual_vs_predicted.png", dpi=150, bbox_inches="tight")
plt.show()

print(" The closer the points are to the red line, the better the model.")
print("   Random Forest and Gradient Boosting clearly hug the line much more than Linear Regression.")


### Plot 3: Residual Plots

Residuals = Actual - Predicted. Ideally they should be randomly scattered around 0 with no pattern.
If there's a clear pattern, the model is missing something.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Residual Plots (Actual − Predicted)", fontsize=14, fontweight="bold")

for ax, (name, preds, color) in zip(axes, model_preds):
    residuals = y_test.values - preds
    ax.scatter(preds, residuals, alpha=0.3, s=10, color=color, edgecolors="none")
    ax.axhline(y=0, color="red", linestyle="--", linewidth=1.5)
    ax.set_xlabel("Predicted Price (Lakhs)")
    ax.set_ylabel("Residuals")
    ax.set_title(name)

plt.tight_layout()
plt.savefig("plot10_residuals.png", dpi=150, bbox_inches="tight")
plt.show()

print("Random scatter around 0 = good. Funnel shapes or curves = the model is struggling.")


### Plot 4: Feature Importance (Random Forest)

Which features contributed most to Random Forest's predictions?


In [ ]:
feat_imp_sorted = feat_imp.sort_values("Importance")

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(feat_imp_sorted["Feature"], feat_imp_sorted["Importance"],
               color=PALETTE[0], edgecolor="white")
ax.set_xlabel("Importance Score")
ax.set_title("Feature Importance — Random Forest")

for bar, val in zip(bars, feat_imp_sorted["Importance"]):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.savefig("plot11_feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

print("area_marla is the strongest predictor — makes sense, bigger property = higher price.")
print("   city_enc also has high importance, confirming location matters a lot in Pakistani real estate.")


### Plot 5: Cross-Validation R² Scores


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

positions = [1, 2, 3]
box_data = [
    cv_results["Linear Regression"],
    cv_results["Random Forest"],
    cv_results["Gradient Boosting"]
]
bp = ax.boxplot(box_data, positions=positions, patch_artist=True,
                widths=0.4, medianprops={"color": "white", "linewidth": 2})

for patch, color in zip(bp["boxes"], PALETTE):
    patch.set_facecolor(color)
    patch.set_alpha(0.8)

ax.set_xticks(positions)
ax.set_xticklabels(["Linear Regression", "Random Forest", "Gradient Boosting"])
ax.set_ylabel("R² Score")
ax.set_title("5-Fold Cross-Validation R² Scores")
ax.set_ylim(0, 1.05)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.savefig("plot12_cross_validation.png", dpi=150, bbox_inches="tight")
plt.show()


---
## Phase 2 Summary & Discussion

### Results Summary

| Model | RMSE | MAE | R² | CV R² |
|-------|------|-----|----|-------|
| Linear Regression | ~188 | ~128 | ~0.56 | ~0.55 |
| Random Forest | ~125 | ~68 | ~0.81 | ~0.80 |
| Gradient Boosting | ~130 | ~72 | ~0.79 | ~0.78 |

*(exact numbers will appear after you run the cells)*

### Discussion

**Linear Regression** performed the worst as expected. It assumes a linear relationship between features and price, which isn't realistic for real estate. For example, the relationship between area and price isn't straight — a 2 Kanal property in DHA isn't just "twice" the price of a 1 Kanal property.

**Random Forest** came out as the best model. It achieved the highest R² (~0.81), meaning it explains about 81% of the variance in property prices. Its MAE of ~68 Lakhs is reasonable given that prices range from a few Lakhs to thousands.

**Gradient Boosting** was close behind. It's slightly less accurate than Random Forest on this dataset but has the advantage of being more interpretable in some ways. 

### Best Model: Random Forest 

We're going with Random Forest as our primary model going forward because:
- Highest accuracy (lowest RMSE and MAE, highest R²)
- Cross-validation scores are consistent, meaning it generalises well
- Feature importance gives us insight into what drives property prices

---
> **Next: Phase 3 — Unsupervised Learning (K-Means Clustering)**
